## U-Net Tutorial: Binary Segmentation of Bean Bucket XCT Images

This notebook is a walkthrough of semantic segmentation using a U-Net style model in PyTorch.

### What problem are we solving?
- **Input**: XCT slice images of a bean bucket
- **Output**: A binary mask per image (`0 = background`, `1 = bean region`)
- **Goal**: Train a model that predicts the mask from the input image

### Why U-Net?
U-Net is a common architecture for segmentation tasks. It combines:
- an **encoder** (downsampling path) that captures context and high-level features
- a **decoder** (upsampling path) that recovers spatial detail
- **skip connections** that pass fine-grained features from encoder to decoder

### Notebook flow
1. Environment setup and device selection
2. Data preview + dataset/dataloader creation
3. U-Net model definition
4. Loss function and training loop
5. Evaluation on test data and visual predictions

Adapted from: https://github.com/usuyama/pytorch-unet

In [ ]:
# Basic setup: make sure the reference U-Net repository is available locally.
# This repo includes helper files (for example, `loss.py`) used later in the notebook.
import os

if not os.path.exists("pytorch_unet.py"):
    # Clone once if the folder is not present yet.
    if not os.path.exists("pytorch_unet"):
        !git clone https://github.com/usuyama/pytorch-unet.git

    # Move into the cloned project directory so local imports work.
    %cd pytorch-unet

In [ ]:
import torch

# Choose GPU if available; otherwise run on CPU.
# For workshop settings, CPU compatibility is useful so everyone can follow along.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Reminder: any model and tensors used for training/inference must be sent to `device`.
# model = model.to(device)
# tensor = tensor.to(device)

## 1) Data Overview and Preprocessing

In this section we:
- inspect a few raw image/mask pairs
- convert masks into binary labels
- create train/validation/test splits
- build PyTorch `DataLoader`s

For first-time learners: segmentation is different from classification because the model predicts a label for **every pixel**, not one label for the whole image.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Paths to input images and binary masks.
IMAGE_DIR = "../BeanBucketImages"
MASK_DIR  = "../BeanBucketMasks_Otsu"

# All images/masks are resized to a common size for batching.
# PIL expects (width, height).
IMG_SIZE  = (256, 256)
NUM_PREVIEW = 3
NUM_CLASSES = 2  # background + foreground

# Read and sort file names so image/mask pairs align by index.
image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith((".tif", ".tiff"))])
mask_files  = sorted([f for f in os.listdir(MASK_DIR)  if f.lower().endswith((".tif", ".tiff"))])
assert len(image_files) == len(mask_files) and len(image_files) > 0

def load_img(path):
    # Convert to 3 channels so shape is always HxWx3.
    return np.array(Image.open(path).convert("RGB").resize(IMG_SIZE))

def load_mask(path):
    # Keep mask single-channel and use nearest-neighbor resize
    # to avoid creating non-integer class values.
    return np.array(Image.open(path).convert("L").resize(IMG_SIZE, Image.NEAREST))

# Preview a few image/mask pairs to sanity-check the dataset.
imgs  = [load_img(os.path.join(IMAGE_DIR, f)) for f in image_files[:NUM_PREVIEW]]
masks = [load_mask(os.path.join(MASK_DIR,  f)) for f in mask_files[:NUM_PREVIEW]]

fig, axes = plt.subplots(NUM_PREVIEW, 2, figsize=(7, 3 * NUM_PREVIEW))
for i in range(NUM_PREVIEW):
    axes[i, 0].imshow(imgs[i])
    axes[i, 0].set_title(image_files[i])
    axes[i, 0].axis("off")

    axes[i, 1].imshow(masks[i], cmap="gray")
    axes[i, 1].set_title(mask_files[i])
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

# Helpful check: for binary masks we expect values like {0, 255} or {0, 1} before binarization.
print("Mask unique values (sample):", np.unique(masks[0]))

In [ ]:
import random
import os
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Fraction of data used for validation and test splits.
VAL_SPLIT  = 0.15
TEST_SPLIT = 0.15

# Number of image/mask pairs sampled from the full dataset.
# Increase this value later for larger training runs.
SAMPLE_N   = 100

class ImageMaskDataset(Dataset):
    """PyTorch Dataset returning (image_tensor, one_hot_mask)."""
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image and convert to RGB array (H, W, 3).
        img = Image.open(self.image_paths[idx]).convert("RGB").resize(IMG_SIZE)
        img = np.array(img)

        # Load mask as grayscale then binarize to class values {0, 1}.
        # Threshold 128 assumes bright pixels are foreground.
        mask = Image.open(self.mask_paths[idx]).convert("L").resize(IMG_SIZE, Image.NEAREST)
        mask = (np.array(mask) > 128).astype(np.uint8)

        # Convert mask to one-hot format expected by our loss/model shape:
        # (NUM_CLASSES, H, W). For binary segmentation this becomes (2, H, W).
        one_hot = np.zeros((NUM_CLASSES, IMG_SIZE[0], IMG_SIZE[1]), dtype=np.float32)
        for c in range(NUM_CLASSES):
            one_hot[c] = (mask == c).astype(np.float32)

        if self.transform:
            img = self.transform(img)

        return img, one_hot

# Discover all files and keep sorted order for deterministic pairing.
all_images = sorted([os.path.join(IMAGE_DIR, f) for f in os.listdir(IMAGE_DIR)])
all_masks  = sorted([os.path.join(MASK_DIR,  f) for f in os.listdir(MASK_DIR)])

assert len(all_images) == len(all_masks), \
    f"Mismatch: {len(all_images)} images vs {len(all_masks)} masks"

# Randomly sample a subset so this notebook runs faster in workshop settings.
random.seed(42)
indices    = random.sample(range(len(all_images)), SAMPLE_N)
all_images = [all_images[i] for i in indices]
all_masks  = [all_masks[i]  for i in indices]

print(f"Randomly selected {SAMPLE_N} pairs")

# Split into train/validation/test.
n       = len(all_images)
n_test  = int(n * TEST_SPLIT)
n_val   = int(n * VAL_SPLIT)
n_train = n - n_val - n_test

train_imgs = all_images[:n_train]
val_imgs   = all_images[n_train : n_train + n_val]
test_imgs  = all_images[n_train + n_val:]

train_msks = all_masks[:n_train]
val_msks   = all_masks[n_train : n_train + n_val]
test_msks  = all_masks[n_train + n_val:]

print(f"Total: {n} | Train: {n_train} | Val: {n_val} | Test: {n_test}")

# Standard image normalization used by many CNN backbones.
trans = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Build datasets and data loaders.
train_set = ImageMaskDataset(train_imgs, train_msks, transform=trans)
val_set   = ImageMaskDataset(val_imgs,   val_msks,   transform=trans)
test_set  = ImageMaskDataset(test_imgs,  test_msks,  transform=trans)

image_datasets = {'train': train_set, 'val': val_set, 'test': test_set}

batch_size = 8

dataloaders = {
    'train': DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=0),
    'val':   DataLoader(val_set,   batch_size=batch_size, shuffle=False, num_workers=0),
    'test':  DataLoader(test_set,  batch_size=batch_size, shuffle=False, num_workers=0),
}

dataset_sizes = {x: len(image_datasets[x]) for x in image_datasets}
print(f"Dataset sizes: {dataset_sizes}")

In [ ]:
import torchvision.utils

def reverse_transform(inp):
    """Undo normalization for visualization."""
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    inp = (inp * 255).astype(np.uint8)
    return inp

# Inspect one batch to verify tensor shapes:
# inputs  -> (B, 3, H, W)
# masks   -> (B, NUM_CLASSES, H, W)
inputs, masks = next(iter(dataloaders['train']))
print("Input batch shape:", inputs.shape)
print("Mask batch shape:", masks.shape)

# Show one sample image from the batch.
plt.imshow(reverse_transform(inputs[3]))
plt.title("Example training image (after reverse normalization)")
plt.axis("off")

## 2) Define the U-Net Model

This implementation uses a **ResNet-18 encoder + U-Net style decoder**.

### Architecture intuition
- The encoder reduces spatial size while learning strong semantic features.
- The decoder upsamples features back to image resolution.
- Skip connections combine coarse semantic information with fine edge/detail information.

Output shape is `(batch, NUM_CLASSES, H, W)`, which means we predict class scores for each pixel.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

def convrelu(in_channels, out_channels, kernel, padding):
    """Convenience block: convolution followed by ReLU activation."""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel, padding=padding),
        nn.ReLU(inplace=True),
    )

class ResNetUNet(nn.Module):
    """ResNet18 encoder + U-Net decoder for segmentation."""
    def __init__(self, n_class):
        super().__init__()

        # Encoder (downsampling path): ResNet18 extracts hierarchical features.
        # weights=None keeps the notebook fully offline-friendly for workshops.
        self.base_model = models.resnet18(weights=None)
        self.base_layers = list(self.base_model.children())

        # Encoder feature maps at multiple resolutions (used for skip connections).
        self.layer0 = nn.Sequential(*self.base_layers[:3])      # (N, 64, H/2,  W/2)
        self.layer0_1x1 = convrelu(64, 64, 1, 0)
        self.layer1 = nn.Sequential(*self.base_layers[3:5])     # (N, 64, H/4,  W/4)
        self.layer1_1x1 = convrelu(64, 64, 1, 0)
        self.layer2 = self.base_layers[5]                       # (N, 128, H/8,  W/8)
        self.layer2_1x1 = convrelu(128, 128, 1, 0)
        self.layer3 = self.base_layers[6]                       # (N, 256, H/16, W/16)
        self.layer3_1x1 = convrelu(256, 256, 1, 0)
        self.layer4 = self.base_layers[7]                       # (N, 512, H/32, W/32)
        self.layer4_1x1 = convrelu(512, 512, 1, 0)

        # Decoder (upsampling path).
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        # Each decoder step concatenates upsampled features with corresponding encoder features.
        self.conv_up3 = convrelu(256 + 512, 512, 3, 1)
        self.conv_up2 = convrelu(128 + 512, 256, 3, 1)
        self.conv_up1 = convrelu(64 + 256, 256, 3, 1)
        self.conv_up0 = convrelu(64 + 256, 128, 3, 1)

        # Early full-resolution branch improves fine boundary reconstruction.
        self.conv_original_size0 = convrelu(3, 64, 3, 1)
        self.conv_original_size1 = convrelu(64, 64, 3, 1)
        self.conv_original_size2 = convrelu(64 + 128, 64, 3, 1)

        # Final 1x1 convolution maps features to per-class logits.
        self.conv_last = nn.Conv2d(64, n_class, 1)

    def forward(self, input):
        # Preserve early high-resolution features from raw input.
        x_original = self.conv_original_size0(input)
        x_original = self.conv_original_size1(x_original)

        # Encoder forward pass.
        layer0 = self.layer0(input)
        layer1 = self.layer1(layer0)
        layer2 = self.layer2(layer1)
        layer3 = self.layer3(layer2)
        layer4 = self.layer4(layer3)

        # Decoder step 1: upsample deepest features and fuse with layer3 skip.
        layer4 = self.layer4_1x1(layer4)
        x = self.upsample(layer4)
        layer3 = self.layer3_1x1(layer3)
        x = torch.cat([x, layer3], dim=1)
        x = self.conv_up3(x)

        # Decoder step 2: fuse with layer2 skip.
        x = self.upsample(x)
        layer2 = self.layer2_1x1(layer2)
        x = torch.cat([x, layer2], dim=1)
        x = self.conv_up2(x)

        # Decoder step 3: fuse with layer1 skip.
        x = self.upsample(x)
        layer1 = self.layer1_1x1(layer1)
        x = torch.cat([x, layer1], dim=1)
        x = self.conv_up1(x)

        # Decoder step 4: fuse with layer0 skip.
        x = self.upsample(x)
        layer0 = self.layer0_1x1(layer0)
        x = torch.cat([x, layer0], dim=1)
        x = self.conv_up0(x)

        # Final high-resolution fusion with early feature branch.
        x = self.upsample(x)
        x = torch.cat([x, x_original], dim=1)
        x = self.conv_original_size2(x)

        # Raw logits (apply sigmoid later for probabilities in binary setup).
        out = self.conv_last(x)
        return out

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import pytorch_unet

# Re-check device so this cell is self-contained if run independently.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)

# Create model for 2-class per-pixel segmentation and move it to selected device.
model = ResNetUNet(NUM_CLASSES)
model = model.to(device)

In [ ]:
model

In [ ]:
from torchsummary import summary
summary(model, input_size=(3, 224, 224))

In [ ]:
from collections import defaultdict
import torch.nn.functional as F
from loss import dice_loss

checkpoint_path = "checkpoint.pth"

def calc_loss(pred, target, metrics, bce_weight=0.5):
    """
    Combined segmentation loss:
    - BCE with logits: pixel-wise classification supervision
    - Dice loss: overlap-focused supervision (helps with class imbalance)
    """
    bce = F.binary_cross_entropy_with_logits(pred, target)

    # Convert logits to probabilities before Dice computation.
    pred = torch.sigmoid(pred)
    dice = dice_loss(pred, target)

    # Weighted sum of the two losses.
    loss = bce * bce_weight + dice * (1 - bce_weight)

    # Track running totals multiplied by batch size for proper epoch averaging.
    metrics['bce'] += bce.data.cpu().numpy() * target.size(0)
    metrics['dice'] += dice.data.cpu().numpy() * target.size(0)
    metrics['loss'] += loss.data.cpu().numpy() * target.size(0)

    return loss

def print_metrics(metrics, epoch_samples, phase):
    """Print average metrics for one epoch phase (train/val/test)."""
    outputs = []
    for k in metrics.keys():
        outputs.append("{}: {:4f}".format(k, metrics[k] / epoch_samples))
    print("{}: {}".format(phase, ", ".join(outputs)))

def train_model(model, optimizer, scheduler, num_epochs=25):
    """
    Standard training loop with validation and best-checkpoint saving.
    Returns the best model (by validation loss) and a history dictionary
    for plotting learning curves.
    """
    best_loss = 1e10

    history = {
        'train_loss': [], 'val_loss': [],
        'train_bce':  [], 'val_bce':  [],
        'train_dice': [], 'val_dice': [],
    }

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        since = time.time()

        # Each epoch has a training phase and a validation phase.
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()   # enable gradient updates
            else:
                model.eval()    # disable dropout/batchnorm updates

            metrics = defaultdict(float)
            epoch_samples = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                # Track gradients only during training phase.
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = calc_loss(outputs, labels, metrics)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                epoch_samples += inputs.size(0)

            print_metrics(metrics, epoch_samples, phase)
            epoch_loss = metrics['loss'] / epoch_samples
            epoch_bce  = metrics['bce']  / epoch_samples
            epoch_dice = metrics['dice'] / epoch_samples

            # Save epoch metrics for plots.
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_bce'].append(epoch_bce)
            history[f'{phase}_dice'].append(epoch_dice)

            if phase == 'train':
                scheduler.step()
                for param_group in optimizer.param_groups:
                    print("LR", param_group['lr'])

            # Keep the best model checkpoint based on validation loss.
            if phase == 'val' and epoch_loss < best_loss:
                print(f"saving best model to {checkpoint_path}")
                best_loss = epoch_loss
                torch.save(model.state_dict(), checkpoint_path)

        time_elapsed = time.time() - since
        print('{:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))

    print('Best val loss: {:4f}'.format(best_loss))

    model.load_state_dict(torch.load(checkpoint_path))
    return model, history

In [ ]:
import torch
import torch.optim as optim
from torch.optim import lr_scheduler
import time

# Fresh model instance for training.
model = ResNetUNet(NUM_CLASSES).to(device)

# Optional transfer-learning strategy: freeze encoder/backbone layers first.
# This can stabilize early training when dataset is limited.
for l in model.base_layers:
    for param in l.parameters():
        param.requires_grad = False

# Optimize only trainable parameters (decoder + unfrozen parts).
optimizer_ft = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# Reduce learning rate every `step_size` epochs.
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=8, gamma=0.1)

# Train and collect history for plotting.
model, history = train_model(model, optimizer_ft, exp_lr_scheduler, num_epochs=20)

In [ ]:
# Plot training/validation curves to diagnose learning behavior.
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Training & Validation Metrics', fontsize=14)

# (history_key_suffix, chart_title, train_color, val_color)
metric_cfg = [
    ('loss', 'Combined Loss (BCE + Dice)', 'tab:blue',   'tab:orange'),
    ('bce',  'Binary Cross-Entropy Loss',  'tab:green',  'tab:red'),
    ('dice', 'Dice Loss',                  'tab:purple', 'tab:brown'),
]

for ax, (key, title, tc, vc) in zip(axes, metric_cfg):
    ax.plot(epochs, history[f'train_{key}'], color=tc, marker='o', label='Train')
    ax.plot(epochs, history[f'val_{key}'],   color=vc, marker='s', linestyle='--', label='Val')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate model on the held-out test split.
# Important: test data is not used for model selection during training.
import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

model.eval()

test_metrics = defaultdict(float)
test_samples = 0

with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        calc_loss(outputs, labels, test_metrics)
        test_samples += inputs.size(0)

print("=== Test Set Results ===")
print_metrics(test_metrics, test_samples, "test")

In [ ]:
# Visualize side-by-side predictions on test samples.
NUM_VISUAL = 4  # number of rows to display

def reverse_transform(inp):
    """Undo normalization so tensors are displayable as images."""
    inp   = inp.numpy().transpose((1, 2, 0))
    mean  = np.array([0.485, 0.456, 0.406])
    std   = np.array([0.229, 0.224, 0.225])
    inp   = std * inp + mean
    inp   = np.clip(inp, 0, 1)
    return (inp * 255).astype(np.uint8)

def masks_to_colorimg(mask):
    """Convert one-hot/probability mask to a black/white RGB visualization."""
    colors = np.array([
        [0,   0,   0],    # class 0 = background (black)
        [255, 255, 255],  # class 1 = foreground (white)
    ], dtype=np.uint8)
    return colors[np.argmax(mask, axis=0)]

model.eval()

# Grab one batch from test loader.
inputs, labels = next(iter(dataloaders['test']))
inputs = inputs.to(device)

with torch.no_grad():
    preds = model(inputs)
    preds = torch.sigmoid(preds).cpu().numpy()

inputs = inputs.cpu()
labels = labels.numpy()

n_show = min(NUM_VISUAL, inputs.size(0))
fig, axes = plt.subplots(n_show, 3, figsize=(10, n_show * 3))
fig.suptitle("Test Predictions\n(Left: Image | Middle: Ground Truth | Right: Prediction)", fontsize=12)

for i in range(n_show):
    axes[i, 0].imshow(reverse_transform(inputs[i]))
    axes[i, 0].set_title("Image")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(masks_to_colorimg(labels[i]))
    axes[i, 1].set_title("Ground Truth")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(masks_to_colorimg(preds[i]))
    axes[i, 2].set_title("Prediction")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

## 3) Suggested Next Experiments

Great job completing a full segmentation pipeline. To improve results, try one change at a time and compare plots/visual outputs:

- **Learning rate**: test values like `1e-3`, `3e-4`, `1e-4`
- **Scheduler**: change `step_size` or try cosine scheduling
- **Loss balance**: adjust `bce_weight` in `calc_loss`
- **Backbone freezing**: unfreeze more encoder layers after a few epochs
- **Batch size / image size**: larger values may improve quality if memory allows

Tip for beginners: keep a small experiment log (settings + final val/test metrics) so you can see what actually helped.